# EYES-DEFY-ANEMIA -- Phase 4 Classification (v2) -- Batch 2

The 2 heaviest architectures (ViT-B/16, ViT-L/16) x 2 tissue types = 4 combos, run as their own dedicated Kaggle session, separate from batch 1's 14 light/medium-architecture combos (`classification-final-fixed.ipynb`). Kept apart specifically because these two are far more compute-expensive than anything in batch 1 (86.6M / 304.3M params vs. under 30M for all 7 batch-1 architectures) -- see `classification/.project_memory/kaggle/01_kaggle_notes.md` for the full batching rationale.

## Setup

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first (this bit an earlier
# version of this notebook during interactive debugging).
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

In [ ]:
# Diagnostic only -- kept for visibility in the saved run log. The path used
# below is already confirmed correct, so this cell doesn't gate anything, but
# it's cheap and makes a future path change easy to spot in the output.
import os

for name in os.listdir("/kaggle/input"):
    path = f"/kaggle/input/{name}"
    print(name, "->", os.listdir(path))

In [ ]:
# Only packages actually missing from Kaggle's base image. Deliberately NOT
# `pip install -r requirements.txt` -- that file is pinned to the local
# Windows/CUDA 13.0 build and would try to reinstall Kaggle's own correctly
# configured GPU PyTorch with an incompatible build.
!pip install -q optuna albumentations

## Data

In [ ]:
import shutil
from pathlib import Path

SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/processed-dataset")
DST_DIR = Path("classification/data/processed")

# Clear the destination first so this cell is fully idempotent -- a re-run (or
# the two messy copy attempts from the earlier notebook version) never leaves
# stale or duplicated content behind. DST_DIR always ends up as an exact,
# deterministic copy of SRC_DIR, nothing more.
shutil.rmtree(DST_DIR, ignore_errors=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

for item in SRC_DIR.iterdir():
    dest = DST_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print("classification/data/processed now contains:")
for sub in sorted(DST_DIR.iterdir()):
    if sub.is_dir():
        n_files = sum(1 for f in sub.rglob("*") if f.is_file())
        print(f"  {sub.name}/  ({n_files} files)")
    else:
        print(f"  {sub.name}")

In [ ]:
import sys

sys.path.insert(0, "classification/scripts")
from dataset import get_dataloaders

loaders = get_dataloaders("palpebral")
images, labels, countries = next(iter(loaders["train"]))
print("Batch shape:", images.shape)
print("Train patients:", len(loaders["train"].dataset))
print("Val patients:", len(loaders["val"].dataset))

In [ ]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidate classification/outputs/{checkpoints,logs,plots}/ into a
    single top-level /kaggle/working/outputs/ folder and re-zip it to
    /kaggle/working/batch2_results.zip. Called after EVERY training cell
    below (all 4 of them), not just at the end -- if the run gets cut short
    partway through (a real possibility, these are the 2 heaviest
    architectures in the whole roster), whatever completed so far is still
    cleanly consolidated and zipped, ready to download. Same design as
    batch 1's sync_outputs() (classification-final-fixed.ipynb), just a
    different output zip name so the two batches' downloads never collide.
    """
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["checkpoints", "logs", "plots"]:
        src = Path("classification/outputs") / sub
        if src.exists():
            shutil.copytree(src, results_dir / sub, dirs_exist_ok=True)
    archive_path = shutil.make_archive("/kaggle/working/batch2_results", "zip", root_dir=str(results_dir))
    n_files = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n_files} files consolidated under {results_dir}, zipped to {archive_path}")


sync_outputs()  # harmless no-op now (nothing produced yet), confirms the function works before training starts

## Training -- Batch 2 (4 combos: ViT-B/16, ViT-L/16, cheapest of the two first)

Each cell is a separate `!python` call, same pattern as batch 1. A failed cell does not halt "Run All" -- check each script's own output (or the saved `classification/outputs/logs/*_study_summary.json` files) after this finishes.

In [ ]:
# Batch 2 / 4 -- ViT-B/16, palpebral
!python classification/v2_scripts/train_vit_b_16_palpebral_v2.py
sync_outputs()

In [ ]:
# Batch 2 / 4 -- ViT-B/16, forniceal_palpebral
!python classification/v2_scripts/train_vit_b_16_forniceal_palpebral_v2.py
sync_outputs()

In [ ]:
# Batch 2 / 4 -- ViT-L/16, palpebral
!python classification/v2_scripts/train_vit_l_16_palpebral_v2.py
sync_outputs()

In [ ]:
# Batch 2 / 4 -- ViT-L/16, forniceal_palpebral
!python classification/v2_scripts/train_vit_l_16_forniceal_palpebral_v2.py
sync_outputs()

## Done -- what to download

Everything is consolidated at `/kaggle/working/outputs/` (checkpoints, logs, plots for whichever combos completed) and zipped to `/kaggle/working/batch2_results.zip`. Both are visible in this notebook version's **Output** tab once you Save Version -> Save & Run All -- download the zip directly from there, or browse the folder for individual files.

In [ ]:
print('Final contents of /kaggle/working/outputs:')
for f in sorted(Path('/kaggle/working/outputs').rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to("/kaggle/working/outputs")}  ({f.stat().st_size / 1e6:.2f} MB)')

zip_path = Path('/kaggle/working/batch2_results.zip')
print(f"\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")